# PIDNet-S Cable Bundle Semantic Segmentation

This Colab notebook trains a PIDNet-S semantic segmentation model for cable bundle analysis.

Classes:

| ID | Class |
|---:|---|
| 0 | background |
| 1 | Black Wire |
| 2 | Lashing Wire |
| 3 | Messenger |

The notebook reads COCO instance annotations, converts them to semantic masks with priority-aware overlap handling, caches the masks on Google Drive, trains with mixed precision, resumes from checkpoints, evaluates on the test split, runs image/directory/video inference, and exports PyTorch, TorchScript, and ONNX artifacts.

In [ ]:
!pip install -q albumentations opencv-python-headless pycocotools tensorboard onnx onnxruntime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

All generated masks, checkpoints, logs, plots, predictions, and exported models are stored on Google Drive so they survive Colab runtime resets.

In [ ]:
import os
import random
import json
import math
import time
import shutil
import inspect
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask
from tqdm.auto import tqdm


DATASET_ROOT = Path('/content/drive/MyDrive/cable_detection/mask_RCNN/augmented_dataset')
TRAIN_IMAGE_DIR = DATASET_ROOT / 'train' / 'images'
TRAIN_ANN_PATH = DATASET_ROOT / 'train' / 'annotations.json'
TEST_IMAGE_DIR = DATASET_ROOT / 'test' / 'images'
TEST_ANN_PATH = DATASET_ROOT / 'test' / 'annotations.json'

RUN_ROOT = Path('/content/drive/MyDrive/cable_detection/pidnet_s_cable_segmentation')
CACHE_ROOT = RUN_ROOT / 'semantic_mask_cache'
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
LOG_DIR = RUN_ROOT / 'tensorboard'
PLOT_DIR = RUN_ROOT / 'plots'
PRED_DIR = RUN_ROOT / 'predictions'
EXPORT_DIR = RUN_ROOT / 'exports'
TEST_RESULT_DIR = RUN_ROOT / 'test_results'

for directory in [CACHE_ROOT, CHECKPOINT_DIR, LOG_DIR, PLOT_DIR, PRED_DIR, EXPORT_DIR, TEST_RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['background', 'Black Wire', 'Lashing Wire', 'Messenger']
NUM_CLASSES = len(CLASS_NAMES)
CLASS_COLORS = np.array([
    [0, 0, 0],
    [40, 40, 40],
    [255, 70, 30],
    [30, 170, 255],
], dtype=np.uint8)

IMG_SIZE = 768
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.15
EARLY_STOP_PATIENCE = 20
SEED = 42
DETERMINISTIC = False
USE_AMP = torch.cuda.is_available()
NUM_WORKERS = 2  # Good starting point for Colab/RTX-class GPUs; set to 0 if workers crash.
PIN_MEMORY = torch.cuda.is_available()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'Run root: {RUN_ROOT}')
if torch.cuda.is_available():
    try:
        torch.set_float32_matmul_precision('high')
        print('Enabled torch float32 matmul precision: high')
    except Exception as exc:
        print(f'Could not set float32 matmul precision: {exc}')

In [ ]:
def seed_everything(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as exc:
            print(f'Deterministic algorithm setting kept permissive: {exc}')
    else:
        torch.backends.cudnn.benchmark = True


seed_everything(SEED, DETERMINISTIC)

## COCO Instance Annotations to Semantic Masks

COCO instance annotations can contain several polygons or RLE masks for a single image. This conversion collapses all instances into one semantic mask per image. When objects overlap, the class priority is:

`background < Messenger < Black Wire < Lashing Wire`

In [ ]:
CLASS_PRIORITY = {
    0: 0,
    3: 1,
    1: 2,
    2: 3,
}

NAME_TO_TRAIN_ID = {
    'black wire': 1,
    'black_wire': 1,
    'blackwire': 1,
    'lashing wire': 2,
    'lashing_wire': 2,
    'lashingwire': 2,
    'messenger': 3,
    'messenger wire': 3,
    'messenger_wire': 3,
}


def normalize_name(name):
    return str(name).strip().lower().replace('-', ' ').replace('__', '_')


def build_category_mapping(coco):
    mapping = {}
    for category in coco.loadCats(coco.getCatIds()):
        category_id = int(category['id'])
        name = normalize_name(category.get('name', category_id))
        compact = name.replace(' ', '')
        underscored = name.replace(' ', '_')
        if name in NAME_TO_TRAIN_ID:
            mapping[category_id] = NAME_TO_TRAIN_ID[name]
        elif underscored in NAME_TO_TRAIN_ID:
            mapping[category_id] = NAME_TO_TRAIN_ID[underscored]
        elif compact in NAME_TO_TRAIN_ID:
            mapping[category_id] = NAME_TO_TRAIN_ID[compact]
        elif category_id in (1, 2, 3):
            mapping[category_id] = category_id
    if not mapping:
        raise ValueError('No usable category mapping found. COCO categories must match the configured cable classes or use ids 1, 2, 3.')
    print('COCO category id -> train id mapping:', mapping)
    return mapping


def coco_file_basename(file_name):
    normalized = str(file_name).replace('\\\\', '/').replace('\\', '/')
    return Path(normalized).name


def resolve_image_path(image_dir, file_name):
    normalized = str(file_name).replace('\\\\', '/').replace('\\', '/')
    basename = coco_file_basename(file_name)
    candidates = [
        image_dir / normalized,
        image_dir / basename,
        image_dir.parent / normalized,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = list(image_dir.rglob(basename))
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f'Image not found for COCO file_name={file_name}. Tried basename={basename} under {image_dir}'
    )


def decode_annotation_mask(annotation, height, width):
    segmentation = annotation.get('segmentation')
    if segmentation:
        if isinstance(segmentation, list):
            rles = coco_mask.frPyObjects(segmentation, height, width)
            rle = coco_mask.merge(rles)
        elif isinstance(segmentation.get('counts'), list):
            rle = coco_mask.frPyObjects(segmentation, height, width)
        else:
            rle = segmentation
        decoded = coco_mask.decode(rle)
        if decoded.ndim == 3:
            decoded = np.any(decoded, axis=2)
        return decoded.astype(bool)

    bbox = annotation.get('bbox')
    if bbox:
        x, y, w, h = bbox
        mask = np.zeros((height, width), dtype=bool)
        x0 = max(0, int(math.floor(x)))
        y0 = max(0, int(math.floor(y)))
        x1 = min(width, int(math.ceil(x + w)))
        y1 = min(height, int(math.ceil(y + h)))
        mask[y0:y1, x0:x1] = True
        return mask

    return np.zeros((height, width), dtype=bool)


def semantic_mask_cache_path(split, image_id, file_name):
    safe_stem = Path(file_name).stem.replace(' ', '_').replace('/', '_').replace('\\', '_')
    return CACHE_ROOT / split / f'{image_id}_{safe_stem}.png'


def convert_coco_split_to_semantic_masks(split, image_dir, annotation_path, force_rebuild=False):
    if not image_dir.exists():
        raise FileNotFoundError(f'Missing image directory: {image_dir}')
    if not annotation_path.exists():
        raise FileNotFoundError(f'Missing annotation file: {annotation_path}')

    coco = COCO(str(annotation_path))
    category_mapping = build_category_mapping(coco)
    split_cache_dir = CACHE_ROOT / split
    split_cache_dir.mkdir(parents=True, exist_ok=True)

    records = []
    image_ids = sorted(coco.getImgIds())
    print(f'Preparing {len(image_ids)} {split} masks in {split_cache_dir}')

    for index, image_id in enumerate(image_ids, start=1):
        image_info = coco.loadImgs([image_id])[0]
        file_name = image_info['file_name']
        image_path = resolve_image_path(image_dir, file_name)
        mask_path = semantic_mask_cache_path(split, image_id, file_name)

        if force_rebuild or not mask_path.exists():
            height = int(image_info.get('height') or 0)
            width = int(image_info.get('width') or 0)
            if height <= 0 or width <= 0:
                image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
                if image is None:
                    raise FileNotFoundError(f'Could not read image: {image_path}')
                height, width = image.shape[:2]

            semantic = np.zeros((height, width), dtype=np.uint8)
            priority_map = np.zeros((height, width), dtype=np.uint8)

            ann_ids = coco.getAnnIds(imgIds=[image_id])
            annotations = coco.loadAnns(ann_ids)
            for annotation in annotations:
                category_id = int(annotation['category_id'])
                if category_id not in category_mapping:
                    continue
                train_id = category_mapping[category_id]
                annotation_mask = decode_annotation_mask(annotation, height, width)
                if not annotation_mask.any():
                    continue
                priority = CLASS_PRIORITY[train_id]
                update_pixels = annotation_mask & (priority >= priority_map)
                semantic[update_pixels] = train_id
                priority_map[update_pixels] = priority

            ok = cv2.imwrite(str(mask_path), semantic)
            if not ok:
                raise IOError(f'Failed to write mask: {mask_path}')

        records.append({
            'image_id': int(image_id),
            'image_path': str(image_path),
            'mask_path': str(mask_path),
            'file_name': file_name,
        })

        if index % 100 == 0 or index == len(image_ids):
            print(f'  {split}: {index}/{len(image_ids)} masks ready')

    if not records:
        raise ValueError(f'No records were created for split={split}')
    return records

## COCO Image Path Resolver Check

Run this cell before mask conversion. It verifies that COCO file names saved with Windows-style paths such as `images\\aug_1_0.jpg` resolve correctly inside Colab/Linux.


In [ ]:
def verify_coco_image_resolution(image_dir, annotation_path):
    coco = COCO(str(annotation_path))
    first_image = coco.loadImgs([coco.getImgIds()[0]])[0]
    file_name = first_image['file_name']
    resolved = resolve_image_path(image_dir, file_name)
    print(f'COCO file_name: {file_name}')
    print(f'Resolved image path: {resolved}')
    if not Path(resolved).exists():
        raise FileNotFoundError(f'Resolver returned a missing path: {resolved}')
    print('Image path resolver check passed.')


verify_coco_image_resolution(TRAIN_IMAGE_DIR, TRAIN_ANN_PATH)


In [ ]:
# Runtime path resolver override for COCO files saved with Windows paths such as images\\aug_1_0.jpg.
def coco_file_basename(file_name):
    normalized = str(file_name).replace('\\\\', '/').replace('\\', '/')
    return Path(normalized).name


def resolve_image_path(image_dir, file_name):
    normalized = str(file_name).replace('\\\\', '/').replace('\\', '/')
    basename = coco_file_basename(file_name)
    candidates = [
        image_dir / normalized,
        image_dir / basename,
        image_dir.parent / normalized,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = list(image_dir.rglob(basename))
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f'Image not found for COCO file_name={file_name}. Tried basename={basename} under {image_dir}'
    )


train_all_records = convert_coco_split_to_semantic_masks('train', TRAIN_IMAGE_DIR, TRAIN_ANN_PATH, force_rebuild=False)
test_records = convert_coco_split_to_semantic_masks('test', TEST_IMAGE_DIR, TEST_ANN_PATH, force_rebuild=False)

rng = random.Random(SEED)
indices = list(range(len(train_all_records)))
rng.shuffle(indices)

if len(indices) >= 2:
    val_count = max(1, int(round(len(indices) * VAL_FRACTION)))
    val_indices = set(indices[:val_count])
else:
    val_indices = set()

train_records = [record for idx, record in enumerate(train_all_records) if idx not in val_indices]
val_records = [record for idx, record in enumerate(train_all_records) if idx in val_indices]
if not val_records:
    val_records = train_records

print(f'Train images: {len(train_records)}')
print(f'Validation images: {len(val_records)}')
print(f'Test images: {len(test_records)}')

## Visual Mask Verification

This cell displays cached semantic masks over the source images before training begins. The overlay uses the fixed class colors defined in the configuration cell.

In [ ]:
def colorize_mask(mask):
    mask = np.asarray(mask, dtype=np.uint8)
    color = CLASS_COLORS[np.clip(mask, 0, NUM_CLASSES - 1)]
    return color


def overlay_mask(image_rgb, mask, alpha=0.45):
    color = colorize_mask(mask)
    overlay = cv2.addWeighted(image_rgb, 1 - alpha, color, alpha, 0)
    return overlay


def show_mask_samples(records, count=4, seed=SEED):
    if not records:
        print('No records available to visualize.')
        return
    rng = random.Random(seed)
    chosen = rng.sample(records, k=min(count, len(records)))
    fig, axes = plt.subplots(len(chosen), 3, figsize=(15, 5 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row, record in enumerate(chosen):
        image_bgr = cv2.imread(record['image_path'], cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(record['image_path'])
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(record['mask_path'], cv2.IMREAD_UNCHANGED)
        if mask is None:
            raise FileNotFoundError(record['mask_path'])

        axes[row, 0].imshow(image_rgb)
        axes[row, 0].set_title(Path(record['image_path']).name)
        axes[row, 1].imshow(colorize_mask(mask))
        axes[row, 1].set_title('Semantic mask')
        axes[row, 2].imshow(overlay_mask(image_rgb, mask))
        axes[row, 2].set_title('Overlay')
        for col in range(3):
            axes[row, col].axis('off')

    legend_handles = [
        plt.Line2D([0], [0], marker='s', color='w', label=name,
                   markerfacecolor=CLASS_COLORS[idx] / 255.0, markersize=12)
        for idx, name in enumerate(CLASS_NAMES)
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=NUM_CLASSES)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()


show_mask_samples(train_all_records, count=4)

## Dataset, Augmentation, and Dataloaders

Training uses the requested Albumentations transforms and resizes images/masks to `768 x 768`.

In [ ]:
def get_train_transform():
    ssr_kwargs = dict(
        shift_limit=0.08,
        scale_limit=0.15,
        rotate_limit=25,
        interpolation=cv2.INTER_LINEAR,
        border_mode=cv2.BORDER_CONSTANT,
        p=0.7,
    )
    ssr_signature = inspect.signature(A.ShiftScaleRotate)
    if 'fill' in ssr_signature.parameters:
        ssr_kwargs['fill'] = (0, 0, 0)
        ssr_kwargs['fill_mask'] = 0
    else:
        ssr_kwargs['value'] = (0, 0, 0)
        ssr_kwargs['mask_value'] = 0

    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_LINEAR),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(**ssr_kwargs),
        A.RandomBrightnessContrast(p=0.35),
        A.CLAHE(p=0.25),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),
        A.RGBShift(r_shift_limit=18, g_shift_limit=18, b_shift_limit=18, p=0.25),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_eval_transform():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_LINEAR),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


class CableSegmentationDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = list(records)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image_bgr = cv2.imread(record['image_path'], cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(record['image_path'])
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(record['mask_path'], cv2.IMREAD_UNCHANGED)
        if mask is None:
            raise FileNotFoundError(record['mask_path'])
        mask = mask.astype(np.int64)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'].long()
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
            mask = torch.from_numpy(mask).long()

        return image, mask


def check_dataset_sample(records, transform, name):
    if not records:
        raise ValueError(f'{name} has no records.')
    dataset = CableSegmentationDataset(records[:1], transform=transform)
    image, mask = dataset[0]
    print(f'{name} sample OK: image={tuple(image.shape)}, mask={tuple(mask.shape)}, classes={sorted(mask.unique().tolist())}')


def make_loader(records, transform, shuffle, batch_size=BATCH_SIZE):
    dataset = CableSegmentationDataset(records, transform=transform)
    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=shuffle and len(dataset) >= batch_size,
    )
    if NUM_WORKERS > 0:
        loader_kwargs['persistent_workers'] = False
        loader_kwargs['prefetch_factor'] = 2
        loader_kwargs['multiprocessing_context'] = 'fork'
    return DataLoader(dataset, **loader_kwargs)


check_dataset_sample(train_records, get_train_transform(), 'Train')
check_dataset_sample(val_records, get_eval_transform(), 'Validation')
check_dataset_sample(test_records, get_eval_transform(), 'Test')

train_loader = make_loader(train_records, get_train_transform(), shuffle=True)
val_loader = make_loader(val_records, get_eval_transform(), shuffle=False)
test_loader = make_loader(test_records, get_eval_transform(), shuffle=False)

batch_images, batch_masks = next(iter(train_loader))
print(f'Batch images: {tuple(batch_images.shape)}')
print(f'Batch masks: {tuple(batch_masks.shape)}')
print(f'Mask values in sample batch: {sorted(batch_masks.unique().tolist())}')

## PIDNet-S Model

This is a compact PyTorch PIDNet-S implementation with proportional/detail/context branches, pixel-attention fusion, a lightweight boundary-guided aggregation block, and a segmentation head.

In [ ]:
class ConvBNAct(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=None, groups=1, activation=True):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU(inplace=True) if activation else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = ConvBNAct(in_channels, out_channels, 3, stride)
        self.conv2 = ConvBNAct(out_channels, out_channels, 3, 1, activation=False)
        if stride != 1 or in_channels != out_channels:
            self.downsample = ConvBNAct(in_channels, out_channels, 1, stride, padding=0, activation=False)
        else:
            self.downsample = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.downsample(x)
        out = self.conv1(x)
        out = self.conv2(out)
        return self.relu(out + identity)


class Bottleneck(nn.Module):
    expansion = 2

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        mid_channels = out_channels // self.expansion
        self.conv1 = ConvBNAct(in_channels, mid_channels, 1, 1, padding=0)
        self.conv2 = ConvBNAct(mid_channels, mid_channels, 3, stride)
        self.conv3 = ConvBNAct(mid_channels, out_channels, 1, 1, padding=0, activation=False)
        if stride != 1 or in_channels != out_channels:
            self.downsample = ConvBNAct(in_channels, out_channels, 1, stride, padding=0, activation=False)
        else:
            self.downsample = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.downsample(x)
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        return self.relu(out + identity)


class PagFM(nn.Module):
    def __init__(self, channels, inter_channels):
        super().__init__()
        self.f_x = ConvBNAct(channels, inter_channels, 1, padding=0, activation=False)
        self.f_y = ConvBNAct(channels, inter_channels, 1, padding=0, activation=False)

    def forward(self, x, y):
        if y.shape[-2:] != x.shape[-2:]:
            y = F.interpolate(y, size=x.shape[-2:], mode='bilinear', align_corners=False)
        sigma = torch.sigmoid(torch.sum(self.f_x(x) * self.f_y(y), dim=1, keepdim=True))
        return sigma * y + (1.0 - sigma) * x


class PAPPM(nn.Module):
    def __init__(self, in_channels, branch_channels, out_channels):
        super().__init__()
        self.scale0 = ConvBNAct(in_channels, branch_channels, 1, padding=0)
        self.scale1 = nn.Sequential(
            nn.AvgPool2d(kernel_size=5, stride=2, padding=2),
            ConvBNAct(in_channels, branch_channels, 1, padding=0),
        )
        self.scale2 = nn.Sequential(
            nn.AvgPool2d(kernel_size=9, stride=4, padding=4),
            ConvBNAct(in_channels, branch_channels, 1, padding=0),
        )
        self.scale3 = nn.Sequential(
            nn.AvgPool2d(kernel_size=17, stride=8, padding=8),
            ConvBNAct(in_channels, branch_channels, 1, padding=0),
        )
        self.scale4 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, branch_channels, 1, bias=True),
            nn.ReLU(inplace=True),
        )
        self.process1 = ConvBNAct(branch_channels, branch_channels, 3)
        self.process2 = ConvBNAct(branch_channels, branch_channels, 3)
        self.process3 = ConvBNAct(branch_channels, branch_channels, 3)
        self.process4 = ConvBNAct(branch_channels, branch_channels, 3)
        self.compression = ConvBNAct(branch_channels * 5, out_channels, 1, padding=0)
        self.shortcut = ConvBNAct(in_channels, out_channels, 1, padding=0)

    def forward(self, x):
        target_size = x.shape[-2:]
        scale0 = self.scale0(x)
        scale1 = self.process1(F.interpolate(self.scale1(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        scale2 = self.process2(F.interpolate(self.scale2(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        scale3 = self.process3(F.interpolate(self.scale3(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        scale4 = self.process4(F.interpolate(self.scale4(x), size=target_size, mode='bilinear', align_corners=False) + scale0)
        out = self.compression(torch.cat([scale0, scale1, scale2, scale3, scale4], dim=1))
        return out + self.shortcut(x)


class LightBag(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.edge_gate = nn.Sequential(
            ConvBNAct(channels, channels, 3),
            nn.Conv2d(channels, channels, 1),
            nn.Sigmoid(),
        )
        self.proj = ConvBNAct(channels, channels, 3)

    def forward(self, p, i, d):
        if i.shape[-2:] != p.shape[-2:]:
            i = F.interpolate(i, size=p.shape[-2:], mode='bilinear', align_corners=False)
        if d.shape[-2:] != p.shape[-2:]:
            d = F.interpolate(d, size=p.shape[-2:], mode='bilinear', align_corners=False)
        gate = self.edge_gate(d)
        fused = gate * p + (1.0 - gate) * i + d
        return self.proj(fused)


class SegmentHead(nn.Module):
    def __init__(self, in_channels, mid_channels, num_classes):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNAct(in_channels, mid_channels, 3),
            nn.Dropout2d(0.1),
            nn.Conv2d(mid_channels, num_classes, 1),
        )

    def forward(self, x, output_size):
        logits = self.block(x)
        return F.interpolate(logits, size=output_size, mode='bilinear', align_corners=False)


class PIDNetS(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        channels = 64
        self.stem = nn.Sequential(
            ConvBNAct(3, 32, 3, stride=2),
            ConvBNAct(32, 32, 3, stride=2),
        )
        self.layer1 = nn.Sequential(BasicBlock(32, 32), BasicBlock(32, 32))
        self.layer2 = nn.Sequential(BasicBlock(32, channels, stride=2), BasicBlock(channels, channels))

        self.p_branch = nn.Sequential(BasicBlock(channels, channels), BasicBlock(channels, channels))
        self.d_branch = nn.Sequential(BasicBlock(channels, channels), BasicBlock(channels, channels))

        self.i_layer3 = nn.Sequential(BasicBlock(channels, 128, stride=2), BasicBlock(128, 128))
        self.i_layer4 = nn.Sequential(BasicBlock(128, 256, stride=2), BasicBlock(256, 256))
        self.context = PAPPM(256, 64, channels)

        self.pag = PagFM(channels, channels // 2)
        self.bag = LightBag(channels)
        self.head = SegmentHead(channels, 128, num_classes)
        self.aux_p = SegmentHead(channels, 64, num_classes)
        self.aux_d = SegmentHead(channels, 64, num_classes)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Conv2d):
            nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        output_size = x.shape[-2:]
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)

        p = self.p_branch(x)
        d = self.d_branch(x)

        i = self.i_layer3(x)
        i = self.i_layer4(i)
        i = self.context(i)
        i = F.interpolate(i, size=p.shape[-2:], mode='bilinear', align_corners=False)

        p = self.pag(p, i)
        fused = self.bag(p, i, d)
        logits = self.head(fused, output_size)

        if self.training:
            return {
                'out': logits,
                'aux_p': self.aux_p(p, output_size),
                'aux_d': self.aux_d(d, output_size),
            }
        return logits


model = PIDNetS(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f'PIDNet-S parameters: {total_params:,} total / {trainable_params:,} trainable')
with torch.no_grad():
    sample_logits = model.eval()(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print(f'Sample output shape: {tuple(sample_logits.shape)}')

## Losses and Metrics

Training optimizes cross entropy plus soft Dice loss. Validation and testing report validation loss, mean IoU, per-class IoU, pixel accuracy, and Dice score.

In [ ]:
def compute_class_weights(records, num_classes=NUM_CLASSES):
    counts = np.zeros(num_classes, dtype=np.float64)
    for record in records:
        mask = cv2.imread(record['mask_path'], cv2.IMREAD_UNCHANGED)
        if mask is None:
            raise FileNotFoundError(record['mask_path'])
        values, value_counts = np.unique(mask, return_counts=True)
        for value, count in zip(values, value_counts):
            if 0 <= int(value) < num_classes:
                counts[int(value)] += int(count)
    counts = np.maximum(counts, 1.0)
    frequencies = counts / counts.sum()
    weights = 1.0 / np.log(1.02 + frequencies)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, target):
        probabilities = torch.softmax(logits, dim=1)
        target_one_hot = F.one_hot(target, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probabilities * target_one_hot, dims)
        cardinality = torch.sum(probabilities + target_one_hot, dims)
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice.mean()


class CombinedSegmentationLoss(nn.Module):
    def __init__(self, class_weights=None, dice_weight=0.5, aux_weight=0.4):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(NUM_CLASSES)
        self.dice_weight = dice_weight
        self.aux_weight = aux_weight

    def main_loss(self, logits, target):
        return self.ce(logits, target) + self.dice_weight * self.dice(logits, target)

    def forward(self, outputs, target):
        if isinstance(outputs, dict):
            loss = self.main_loss(outputs['out'], target)
            loss = loss + self.aux_weight * self.ce(outputs['aux_p'], target)
            loss = loss + self.aux_weight * self.ce(outputs['aux_d'], target)
            return loss
        return self.main_loss(outputs, target)


def update_confusion_matrix(confusion, preds, targets, num_classes=NUM_CLASSES):
    preds = preds.detach().view(-1).to(torch.int64)
    targets = targets.detach().view(-1).to(torch.int64)
    valid = (targets >= 0) & (targets < num_classes)
    labels = num_classes * targets[valid] + preds[valid]
    bincount = torch.bincount(labels, minlength=num_classes ** 2)
    confusion += bincount.reshape(num_classes, num_classes).cpu()
    return confusion


def metrics_from_confusion(confusion):
    confusion = confusion.float()
    tp = torch.diag(confusion)
    row_sum = confusion.sum(dim=1)
    col_sum = confusion.sum(dim=0)
    union = row_sum + col_sum - tp
    iou = tp / torch.clamp(union, min=1.0)
    dice = (2.0 * tp) / torch.clamp(row_sum + col_sum, min=1.0)
    pixel_accuracy = tp.sum() / torch.clamp(confusion.sum(), min=1.0)
    mean_iou = iou.mean()
    mean_dice = dice.mean()
    return {
        'iou': iou.numpy(),
        'dice': dice.numpy(),
        'pixel_accuracy': float(pixel_accuracy.item()),
        'mean_iou': float(mean_iou.item()),
        'mean_dice': float(mean_dice.item()),
    }


class_weights = compute_class_weights(train_records).to(DEVICE)
criterion = CombinedSegmentationLoss(class_weights=class_weights)
print('Class weights:', {CLASS_NAMES[idx]: round(float(weight), 4) for idx, weight in enumerate(class_weights.cpu())})

## Training, Resume, and Checkpoints

The training cell automatically resumes from `latest.pth` when it exists. Checkpoints include model, optimizer, scheduler, AMP scaler, epoch, best scores, early-stopping counter, and history.

In [ ]:
LATEST_CKPT = CHECKPOINT_DIR / 'latest.pth'
BEST_MIOU_CKPT = CHECKPOINT_DIR / 'best_miou.pth'
BEST_LOSS_CKPT = CHECKPOINT_DIR / 'best_loss.pth'


def train_one_epoch(model, loader, optimizer, criterion, scaler, device, epoch=None, total_epochs=None):
    model.train()
    running_loss = 0.0
    seen = 0
    total_batches = len(loader)
    epoch_text = f'{epoch}/{total_epochs}' if epoch is not None and total_epochs is not None else str(epoch or '')
    progress = tqdm(
        loader,
        total=total_batches,
        desc=f'Train epoch {epoch_text}',
        leave=True,
        dynamic_ncols=True,
        unit='batch',
    )
    for batch_idx, (images, masks) in enumerate(progress, start=1):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = images.size(0)
        running_loss += float(loss.detach().item()) * batch_size
        seen += batch_size
        progress.set_postfix(
            batch=f'{batch_idx}/{total_batches}',
            loss=f'{running_loss / max(seen, 1):.4f}',
            lr=f'{optimizer.param_groups[0]["lr"]:.2e}',
        )
    return running_loss / max(seen, 1)


@torch.no_grad()
def evaluate(model, loader, criterion, device, desc='Validation'):
    model.eval()
    running_loss = 0.0
    seen = 0
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    total_batches = len(loader)
    progress = tqdm(
        loader,
        total=total_batches,
        desc=desc,
        leave=True,
        dynamic_ncols=True,
        unit='batch',
    )
    for batch_idx, (images, masks) in enumerate(progress, start=1):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, masks)
        preds = torch.argmax(logits, dim=1)
        update_confusion_matrix(confusion, preds.cpu(), masks.cpu())
        partial_metrics = metrics_from_confusion(confusion)
        batch_size = images.size(0)
        running_loss += float(loss.detach().item()) * batch_size
        seen += batch_size
        progress.set_postfix(
            batch=f'{batch_idx}/{total_batches}',
            loss=f'{running_loss / max(seen, 1):.4f}',
            mIoU=f'{partial_metrics["mean_iou"]:.4f}',
            pix_acc=f'{partial_metrics["pixel_accuracy"]:.4f}',
        )

    metrics = metrics_from_confusion(confusion)
    metrics['loss'] = running_loss / max(seen, 1)
    metrics['confusion'] = confusion.numpy()
    return metrics


def save_checkpoint(path, epoch, model, optimizer, scheduler, scaler, best_miou, best_loss, epochs_without_miou_gain, history):
    checkpoint = {
        'epoch': int(epoch),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'best_miou': float(best_miou),
        'best_loss': float(best_loss),
        'epochs_without_miou_gain': int(epochs_without_miou_gain),
        'history': history,
        'class_names': CLASS_NAMES,
        'img_size': IMG_SIZE,
    }
    torch.save(checkpoint, str(path))


def load_checkpoint(path, map_location=DEVICE):
    try:
        return torch.load(str(path), map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=map_location)


def load_latest_checkpoint(model, optimizer, scheduler, scaler):
    if not LATEST_CKPT.exists():
        return 0, -1.0, float('inf'), 0, defaultdict(list)
    checkpoint = load_checkpoint(LATEST_CKPT, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = int(checkpoint['epoch']) + 1
    best_miou = float(checkpoint.get('best_miou', -1.0))
    best_loss = float(checkpoint.get('best_loss', float('inf')))
    epochs_without_miou_gain = int(checkpoint.get('epochs_without_miou_gain', 0))
    history = defaultdict(list, checkpoint.get('history', {}))
    print(f'Resumed from {LATEST_CKPT} at epoch {start_epoch}')
    return start_epoch, best_miou, best_loss, epochs_without_miou_gain, history


print('PIDNet training cell version: nonpersistent-workers-v5')
print(f'Training speed settings: batch_size={BATCH_SIZE}, img_size={IMG_SIZE}, num_workers={NUM_WORKERS}, amp={USE_AMP}, deterministic={DETERMINISTIC}')

def shutdown_existing_loader_workers(*loader_names):
    for loader_name in loader_names:
        loader = globals().get(loader_name)
        iterator = getattr(loader, '_iterator', None)
        if iterator is not None and hasattr(iterator, '_shutdown_workers'):
            try:
                iterator._shutdown_workers()
                print(f'Shut down old workers for {loader_name}.')
            except Exception as exc:
                print(f'Could not shut down old workers for {loader_name}: {exc}')


shutdown_existing_loader_workers('train_loader', 'val_loader', 'test_loader')
print(f'Rebuilding DataLoaders with num_workers={NUM_WORKERS}, persistent_workers=False.')
train_loader = make_loader(train_records, get_train_transform(), shuffle=True)
val_loader = make_loader(val_records, get_eval_transform(), shuffle=False)
test_loader = make_loader(test_records, get_eval_transform(), shuffle=False)
print(f'DataLoader workers: train={train_loader.num_workers}, val={val_loader.num_workers}, test={test_loader.num_workers}')

model = PIDNetS(num_classes=NUM_CLASSES).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LEARNING_RATE * 0.02)
scaler = GradScaler('cuda', enabled=USE_AMP)
writer = SummaryWriter(log_dir=str(LOG_DIR))

start_epoch, best_miou, best_loss, epochs_without_miou_gain, history = load_latest_checkpoint(
    model, optimizer, scheduler, scaler
)

if start_epoch >= EPOCHS:
    print(f'Checkpoint already reached epoch {start_epoch}. Increase EPOCHS to continue training.')
else:
    epoch_progress = tqdm(
        range(start_epoch, EPOCHS),
        total=EPOCHS - start_epoch,
        desc='Overall training progress',
        leave=True,
        dynamic_ncols=True,
        unit='epoch',
    )
    for epoch in epoch_progress:
        epoch_start = time.time()
        current_epoch = epoch + 1
        print(f'\\n===== Epoch {current_epoch}/{EPOCHS} =====')
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, DEVICE,
            epoch=current_epoch, total_epochs=EPOCHS,
        )
        val_metrics = evaluate(
            model, val_loader, criterion, DEVICE,
            desc=f'Validation epoch {current_epoch}/{EPOCHS}',
        )
        scheduler.step()
        epoch_progress.set_postfix(
            train_loss=f'{train_loss:.4f}',
            val_loss=f'{val_metrics["loss"]:.4f}',
            mIoU=f'{val_metrics["mean_iou"]:.4f}',
            pix_acc=f'{val_metrics["pixel_accuracy"]:.4f}',
        )

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['mean_iou'].append(val_metrics['mean_iou'])
        history['pixel_accuracy'].append(val_metrics['pixel_accuracy'])
        history['mean_dice'].append(val_metrics['mean_dice'])
        history['lr'].append(float(optimizer.param_groups[0]['lr']))

        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_metrics['loss'], epoch)
        writer.add_scalar('Metrics/mIoU', val_metrics['mean_iou'], epoch)
        writer.add_scalar('Metrics/pixel_accuracy', val_metrics['pixel_accuracy'], epoch)
        writer.add_scalar('Metrics/dice', val_metrics['mean_dice'], epoch)
        writer.add_scalar('LearningRate', optimizer.param_groups[0]['lr'], epoch)
        for class_index, class_name in enumerate(CLASS_NAMES):
            writer.add_scalar(f'IoU/{class_name}', float(val_metrics['iou'][class_index]), epoch)

        improved_miou = val_metrics['mean_iou'] > best_miou
        improved_loss = val_metrics['loss'] < best_loss

        if improved_miou:
            best_miou = val_metrics['mean_iou']
            epochs_without_miou_gain = 0
        else:
            epochs_without_miou_gain += 1
        if improved_loss:
            best_loss = val_metrics['loss']

        save_checkpoint(LATEST_CKPT, epoch, model, optimizer, scheduler, scaler, best_miou, best_loss, epochs_without_miou_gain, dict(history))
        if improved_miou:
            shutil.copy2(LATEST_CKPT, BEST_MIOU_CKPT)
        if improved_loss:
            shutil.copy2(LATEST_CKPT, BEST_LOSS_CKPT)

        elapsed = time.time() - epoch_start
        per_class_iou = ', '.join(f'{CLASS_NAMES[i]}={val_metrics["iou"][i]:.4f}' for i in range(NUM_CLASSES))
        print(
            f'Epoch {epoch + 1:03d}/{EPOCHS} | '
            f'train_loss={train_loss:.4f} | val_loss={val_metrics["loss"]:.4f} | '
            f'mIoU={val_metrics["mean_iou"]:.4f} | pixel_acc={val_metrics["pixel_accuracy"]:.4f} | '
            f'dice={val_metrics["mean_dice"]:.4f} | lr={optimizer.param_groups[0]["lr"]:.6g} | '
            f'{elapsed:.1f}s'
        )
        print(f'  Per-class IoU: {per_class_iou}')

        if epochs_without_miou_gain >= EARLY_STOP_PATIENCE:
            print(f'Early stopping after {EARLY_STOP_PATIENCE} epochs without validation mIoU improvement.')
            break

writer.flush()
writer.close()
print(f'Latest checkpoint: {LATEST_CKPT}')
print(f'Best mIoU checkpoint: {BEST_MIOU_CKPT}')
print(f'Best loss checkpoint: {BEST_LOSS_CKPT}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$LOG_DIR"

## Training Curves

The plots are saved to Google Drive and displayed inline.

In [ ]:
def load_history_from_latest():
    if not LATEST_CKPT.exists():
        raise FileNotFoundError(f'No checkpoint found at {LATEST_CKPT}')
    checkpoint = load_checkpoint(LATEST_CKPT, map_location='cpu')
    return checkpoint.get('history', {})


history_for_plot = load_history_from_latest()
epochs_axis = np.arange(1, len(history_for_plot.get('train_loss', [])) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

plot_specs = [
    ('train_loss', 'Training Loss'),
    ('val_loss', 'Validation Loss'),
    ('mean_iou', 'Mean IoU'),
    ('pixel_accuracy', 'Pixel Accuracy'),
]

for ax, (key, title) in zip(axes, plot_specs):
    values = history_for_plot.get(key, [])
    ax.plot(epochs_axis[:len(values)], values, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = PLOT_DIR / 'training_curves.png'
fig.savefig(plot_path, dpi=160)
plt.show()
print(f'Saved training curves to {plot_path}')

## Test Evaluation

This cell evaluates the best validation-mIoU checkpoint on the held-out test set and saves a confusion matrix plus a JSON metrics report.

In [ ]:
def load_model_checkpoint_for_eval(checkpoint_path=BEST_MIOU_CKPT):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')
    model = PIDNetS(num_classes=NUM_CLASSES).to(DEVICE)
    checkpoint = load_checkpoint(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    return model, checkpoint


test_model, test_checkpoint = load_model_checkpoint_for_eval(BEST_MIOU_CKPT)
test_metrics = evaluate(test_model, test_loader, criterion, DEVICE)

test_report = {
    'checkpoint': str(BEST_MIOU_CKPT),
    'loss': float(test_metrics['loss']),
    'pixel_accuracy': float(test_metrics['pixel_accuracy']),
    'mIoU': float(test_metrics['mean_iou']),
    'mean_dice': float(test_metrics['mean_dice']),
    'per_class_iou': {CLASS_NAMES[i]: float(test_metrics['iou'][i]) for i in range(NUM_CLASSES)},
    'per_class_dice': {CLASS_NAMES[i]: float(test_metrics['dice'][i]) for i in range(NUM_CLASSES)},
}

report_path = TEST_RESULT_DIR / 'test_metrics.json'
with open(report_path, 'w') as f:
    json.dump(test_report, f, indent=2)

confusion = test_metrics['confusion']
confusion_path = TEST_RESULT_DIR / 'confusion_matrix.npy'
np.save(confusion_path, confusion)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(confusion, cmap='Blues')
ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_yticks(np.arange(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted')
ax.set_ylabel('Ground Truth')
ax.set_title('Test Confusion Matrix')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, int(confusion[i, j]), ha='center', va='center', color='black', fontsize=9)
fig.colorbar(im, ax=ax)
plt.tight_layout()
confusion_png_path = TEST_RESULT_DIR / 'confusion_matrix.png'
fig.savefig(confusion_png_path, dpi=160)
plt.show()

print(json.dumps(test_report, indent=2))
print(f'Saved test metrics to {report_path}')
print(f'Saved confusion matrix array to {confusion_path}')
print(f'Saved confusion matrix plot to {confusion_png_path}')

## Inference Utilities

These functions load the best checkpoint, preprocess images to `768 x 768`, run PIDNet-S, restore predictions to the original image size, and save color masks plus overlays.

In [ ]:
INFER_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
INFER_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_image_for_inference(image_rgb):
    original_size = image_rgb.shape[:2]
    resized = cv2.resize(image_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    image = resized.astype(np.float32) / 255.0
    image = (image - INFER_MEAN) / INFER_STD
    tensor = torch.from_numpy(image.transpose(2, 0, 1)).unsqueeze(0).float()
    return tensor, original_size


@torch.no_grad()
def predict_mask(model, image_rgb, device=DEVICE):
    tensor, original_size = preprocess_image_for_inference(image_rgb)
    tensor = tensor.to(device)
    with autocast('cuda', enabled=USE_AMP):
        logits = model(tensor)
    pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
    pred = cv2.resize(pred, (original_size[1], original_size[0]), interpolation=cv2.INTER_NEAREST)
    return pred


def save_prediction_outputs(image_rgb, pred_mask, output_stem, output_dir=PRED_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    color_mask = colorize_mask(pred_mask)
    overlay = overlay_mask(image_rgb, pred_mask, alpha=0.45)

    mask_path = output_dir / f'{output_stem}_mask.png'
    color_path = output_dir / f'{output_stem}_color.png'
    overlay_path = output_dir / f'{output_stem}_overlay.png'

    cv2.imwrite(str(mask_path), pred_mask)
    cv2.imwrite(str(color_path), cv2.cvtColor(color_mask, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(overlay_path), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    return mask_path, color_path, overlay_path


inference_model, _ = load_model_checkpoint_for_eval(BEST_MIOU_CKPT)
print('Inference model loaded.')

## Single Image Inference

The default image is the first test image. Change `SINGLE_IMAGE_PATH` to run another image.

In [ ]:
SINGLE_IMAGE_PATH = test_records[0]['image_path'] if test_records else ''

if SINGLE_IMAGE_PATH and Path(SINGLE_IMAGE_PATH).exists():
    image_bgr = cv2.imread(SINGLE_IMAGE_PATH, cv2.IMREAD_COLOR)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pred_mask = predict_mask(inference_model, image_rgb)
    output_paths = save_prediction_outputs(image_rgb, pred_mask, Path(SINGLE_IMAGE_PATH).stem, PRED_DIR / 'single_image')

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image_rgb)
    axes[0].set_title('Image')
    axes[1].imshow(colorize_mask(pred_mask))
    axes[1].set_title('Prediction')
    axes[2].imshow(overlay_mask(image_rgb, pred_mask))
    axes[2].set_title('Overlay')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    print('Saved:', [str(path) for path in output_paths])
else:
    print('Single image path is empty or unavailable.')

## Directory Inference

This cell runs inference for every image in a directory and saves grayscale masks, color masks, and overlays.

In [ ]:
DIRECTORY_INPUT_PATH = str(TEST_IMAGE_DIR)
DIRECTORY_OUTPUT_PATH = PRED_DIR / 'directory'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

directory_input = Path(DIRECTORY_INPUT_PATH)
if directory_input.exists():
    image_paths = sorted(path for path in directory_input.rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS)
    print(f'Found {len(image_paths)} images in {directory_input}')
    for image_path in image_paths:
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            print(f'Skipping unreadable image: {image_path}')
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        pred_mask = predict_mask(inference_model, image_rgb)
        relative_stem = '_'.join(image_path.relative_to(directory_input).with_suffix('').parts)
        save_prediction_outputs(image_rgb, pred_mask, relative_stem, DIRECTORY_OUTPUT_PATH)
    print(f'Directory predictions saved to {DIRECTORY_OUTPUT_PATH}')
else:
    print(f'Directory does not exist: {directory_input}')

## Video Inference

Set `VIDEO_INPUT_PATH` to a video file on Drive. The cell writes an overlay video to `VIDEO_OUTPUT_PATH`.

In [ ]:
VIDEO_INPUT_PATH = '/content/drive/MyDrive/cable_detection/video_input.mp4'
VIDEO_OUTPUT_PATH = str(PRED_DIR / 'video_overlay.mp4')

video_input = Path(VIDEO_INPUT_PATH)
if video_input.exists():
    capture = cv2.VideoCapture(str(video_input))
    if not capture.isOpened():
        raise IOError(f'Could not open video: {video_input}')

    fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(VIDEO_OUTPUT_PATH, fourcc, fps, (width, height))
    if not writer.isOpened():
        raise IOError(f'Could not create video writer: {VIDEO_OUTPUT_PATH}')

    frame_index = 0
    while True:
        ok, frame_bgr = capture.read()
        if not ok:
            break
        image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pred_mask = predict_mask(inference_model, image_rgb)
        overlay_rgb = overlay_mask(image_rgb, pred_mask, alpha=0.45)
        writer.write(cv2.cvtColor(overlay_rgb, cv2.COLOR_RGB2BGR))
        frame_index += 1
        if frame_index % 50 == 0:
            print(f'Processed {frame_index} frames')

    capture.release()
    writer.release()
    print(f'Video inference complete. Saved {frame_index} frames to {VIDEO_OUTPUT_PATH}')
else:
    print(f'Video file not found: {video_input}')

## Export

Exports the best mIoU PyTorch checkpoint, a TorchScript model, and an ONNX model to Google Drive.

In [ ]:
import onnx

export_model, export_checkpoint = load_model_checkpoint_for_eval(BEST_MIOU_CKPT)
export_model.eval()

best_copy_path = EXPORT_DIR / 'best_miou.pth'
shutil.copy2(BEST_MIOU_CKPT, best_copy_path)

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

torchscript_path = EXPORT_DIR / 'pidnet_s_cable_segmentation_torchscript.pt'
with torch.no_grad():
    traced = torch.jit.trace(export_model, dummy_input)
    traced.save(str(torchscript_path))

onnx_path = EXPORT_DIR / 'pidnet_s_cable_segmentation.onnx'
torch.onnx.export(
    export_model,
    dummy_input,
    str(onnx_path),
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
)
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print(f'Exported PyTorch checkpoint: {best_copy_path}')
print(f'Exported TorchScript model: {torchscript_path}')
print(f'Exported ONNX model: {onnx_path}')